# 301 · Two schema cultures experiment

Companion to [Two schema cultures](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/301/two-schema-cultures/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/301/two_schema_cultures.ipynb)

**Goal:** contrast **field-number culture** (Protobuf-class skip unknowns) with a toy **resolution culture** (writer/reader schemas + defaults)—operations, not speed ranking.

> **Honesty banner:** notebook experiments implement the article **Experiments** blocks. They do not replace suite Results. Timings/sizes are illustrative.



In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Dict, Tuple


def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def decode_varint(buf: bytes, i: int = 0) -> Tuple[int, int]:
    value = shift = 0
    while True:
        b = buf[i]; i += 1
        value |= (b & 0x7F) << shift
        if b < 0x80:
            return value, i
        shift += 7


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)



## Field-number culture: add field 3, old reader skips



In [ ]:
# Writer v2 encodes id=1, name=2, email=3
def encode_pb_v2(id_: int, name: str, email: str) -> bytes:
    out = bytearray()
    out += encode_key(1, 0) + encode_varint(id_)
    nb = name.encode(); out += encode_key(2, 2) + encode_varint(len(nb)) + nb
    eb = email.encode(); out += encode_key(3, 2) + encode_varint(len(eb)) + eb
    return bytes(out)


def decode_pb_v1(buf: bytes) -> dict:
    i = 0
    out = {"id": 0, "name": ""}
    while i < len(buf):
        key, i = decode_varint(buf, i)
        fn, wt = key >> 3, key & 7
        if wt == 0:
            v, i = decode_varint(buf, i)
            if fn == 1:
                out["id"] = v
        elif wt == 2:
            n, i = decode_varint(buf, i)
            payload = buf[i : i + n]; i += n
            if fn == 2:
                out["name"] = payload.decode()
            # unknown (e.g. email=3): skip
        else:
            raise ValueError(wt)
    return out


wire = encode_pb_v2(1, "Ada", "ada@ex.com")
print("v1 reader view:", decode_pb_v1(wire))
print("Culture rule: never reuse field number 3 for a new meaning later.")



## Resolution culture (toy): writer schema + reader schema + defaults



In [ ]:
@dataclass
class Field:
    name: str
    typ: str
    default: Any = None


def resolve(writer_schema: list[Field], reader_schema: list[Field], writer_data: Dict[str, Any]) -> Dict[str, Any]:
    """Minimal name-based projection with defaults (Avro-class intuition only)."""
    wset = {f.name: f for f in writer_schema}
    result = {}
    for rf in reader_schema:
        if rf.name in writer_data:
            result[rf.name] = writer_data[rf.name]
        elif rf.default is not None or rf.name not in wset:
            result[rf.name] = rf.default
        else:
            raise KeyError(f"missing required without default: {rf.name}")
    return result


writer_v2 = [Field("id", "int"), Field("name", "string"), Field("email", "string", default="")]
reader_v1 = [Field("id", "int"), Field("name", "string")]
reader_v2 = [Field("id", "int"), Field("name", "string"), Field("email", "string", default="")]

payload_v2 = {"id": 1, "name": "Ada", "email": "ada@ex.com"}
payload_v1 = {"id": 1, "name": "Ada"}

print("writer v2 → reader v1:", resolve(writer_v2, reader_v1, payload_v2))
print("writer v1 → reader v2:", resolve(writer_v1 := writer_v2[:2], reader_v2, payload_v1))
print("Registry would enforce BACKWARD/FORWARD/FULL on schema subjects—not shown here.")



## Decision frame

| Need independent producers + registry gates | → resolution culture |
| Shared IDL repo + codegen RPC | → field-number culture |

Neither is “more schema-driven.” They differ in **control plane**.

